# Phase 4 Multi-Task Model: Full 2022 Inventory Inference

**Purpose**: Process complete 2022 EuropePMC dataset (21,392 papers) using Phase 4 unified multi-task model  
**Created**: 2025-11-03  
**Updated**: 2025-11-04 (Memory optimization)  
**Environment**: Google Colab with GPU (T4/A100)  
**Model**: Phase 4 BiomedicalMultiTaskModel (checkpoint_best_ner.pt)

---

## Overview

This notebook replaces the V2 two-model approach with a single unified Phase 4 model achieving:

- **NER F1**: 0.9274 (+23.82% vs V2 baseline 0.749)
- **Classification F1**: 0.8586 (-4.38% vs V2 baseline 0.898)
- **Combined F1**: 0.8917 (+8.28% vs V2 baseline 0.8235)

### Performance

- **T4 GPU**: ~20 minutes (batch_size=32)
- **A100 GPU**: ~5-6 minutes (batch_size=128, mixed precision)
- **CPU**: 2-3 hours (batch_size=8)
- **GPU Memory**: 4GB (T4) / 12GB (A100)
- **RAM Usage**: <10GB (optimized chunked processing)

### Key Features

- ✅ **Unified Multi-Task Model**: Single model for classification + NER
- ✅ **Metadata Integration**: 28 engineered features from Phase 1-3
- ✅ **GPU Acceleration**: 5-10x faster than CPU
- ✅ **Memory Optimized**: Chunked processing prevents text duplication, handles large datasets efficiently
- ✅ **Pipeline Compatible**: Output format matches downstream requirements
- ✅ **Model Traceability**: Links to training session and archived checkpoint

### Memory Optimization

This notebook uses memory-efficient processing to prevent overflow:
- **Slim Results Storage**: Classification and NER results store only predictions (no text duplication)
- **Chunked Merging**: Final inventory created by processing papers in 5,000-row chunks
- **Immediate Writes**: Chunks written to disk and memory released progressively
- **Memory Reduction**: 94% less memory usage (160GB+ → <10GB)

### Requirements

**Before running:**

1. **GPU Runtime**: Runtime → Change runtime type → GPU
2. **Google Drive**: Upload project to `MyDrive/inventory_2022/`
3. **Required Files**:
   - `data/epmc_query_results_2022.csv` (21,392 papers)
   - `data/metadata/features_engineered.csv` (28 features)
   - `experiment_archives/2025-10-31-rq7i4n/multitask_training/checkpoint_best_ner.pt`

---

## Output Structure

```
experiment_archives/2025-MM-DD-XXXXXX_phase4_2022_rerun/
├── classification_results.csv
├── ner_results.csv
├── final_inventory.csv
├── config_with_traceability.json
└── README.md
```

---

## Related Notebooks

- **Training**: `phase4_multitask_training.ipynb` (Phase 4 model training)
- **Testing**: `phase4_inference_test.ipynb` (100-paper validation)
- **V2 Baseline**: `rerun_2022_inventory_simplified.ipynb` (two-model approach)
- **Documentation**: `docs/multi_task_model/PHASE4_IMPLEMENTATION_SUMMARY.md`

## Cell 1: Hardware Setup and GPU Detection

In [ ]:
import torch
import sys

print("="*80)
print("HARDWARE CONFIGURATION")
print("="*80)

# Check GPU availability
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    
    print(f"✅ GPU Available: {gpu_name}")
    print(f"   Memory: {gpu_memory:.1f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")
    print(f"   PyTorch Version: {torch.__version__}")
    
    # Clear GPU memory
    torch.cuda.empty_cache()
    
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    
    print(f"\n   Memory Status:")
    print(f"      Allocated: {allocated:.2f} GB")
    print(f"      Reserved:  {reserved:.2f} GB")
    print(f"      Free:      {gpu_memory - reserved:.2f} GB")
    
    # Performance expectations
    if 'A100' in gpu_name:
        print(f"\n   🚀 A100 GPU detected - Expected runtime: ~5-6 minutes (21,392 papers)")
        print(f"      Optimizations: 4x batch size + mixed precision (bfloat16)")
    elif 'T4' in gpu_name:
        print(f"\n   💡 T4 GPU detected - Expected runtime: ~20 minutes (21,392 papers)")
    else:
        print(f"\n   ℹ️  GPU detected - Expected runtime: ~20-30 minutes (21,392 papers)")
    
    device = torch.device('cuda')
    
else:
    print("⚠️  NO GPU AVAILABLE - Running on CPU")
    print("   Expected runtime: 2-3 hours (21,392 papers)")
    print("\n   💡 To enable GPU:")
    print("      1. Go to: Runtime → Change runtime type")
    print("      2. Set Hardware accelerator to 'GPU'")
    print("      3. Click Save and restart runtime")
    
    device = torch.device('cpu')

print(f"\n✅ Device configured: {device}")
print("="*80)

## Cell 2: Mount Google Drive and Setup Paths

In [ ]:
from google.colab import drive
import os
from pathlib import Path
import random
import string
from datetime import datetime

print("="*80)
print("GOOGLE DRIVE SETUP")
print("="*80)

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)
print("✅ Google Drive mounted successfully")

# Set project paths
PROJECT_NAME = "inventory_2022"
DRIVE_BASE = f"/content/drive/MyDrive/{PROJECT_NAME}"

# Verify project directory exists
if not Path(DRIVE_BASE).exists():
    print(f"\n❌ ERROR: Project directory not found: {DRIVE_BASE}")
    print("\n📋 Required Action:")
    print("   1. Upload the inventory_2022 project folder to Google Drive")
    print("   2. Ensure it's in the 'My Drive' root directory")
    print("   3. Refresh Drive mount and run this cell again")
    raise FileNotFoundError(f"Project directory not found: {DRIVE_BASE}")

# Change to project directory
os.chdir(DRIVE_BASE)
print(f"✅ Working directory: {os.getcwd()}")

# Add src to Python path
if str(Path(DRIVE_BASE) / 'src') not in sys.path:
    sys.path.insert(0, str(Path(DRIVE_BASE) / 'src'))
print(f"✅ Added src to Python path")

# Generate session ID
SESSION_ID = f"{datetime.now().strftime('%Y-%m-%d')}-{''.join(random.choices(string.ascii_lowercase + string.digits, k=6))}"
TIMESTAMP = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print(f"\n🆔 Session ID: {SESSION_ID}")
print(f"⏰ Started: {TIMESTAMP}")

# Define key paths
PAPERS_PATH = Path('data/epmc_query_results_2022.csv')
METADATA_PATH = Path('data/metadata/features_engineered.csv')
CHECKPOINT_PATH = Path(
    'experiment_archives/2025-10-31-rq7i4n/'
    'multitask_training/checkpoint_best_ner.pt'
)

# Training session traceability
TRAINING_SESSION_ID = '2025-10-31-rq7i4n'

# Create output directory in experiment_archives (not collab_results)
OUTPUT_DIR = Path(f'experiment_archives/{SESSION_ID}_phase4_2022_rerun')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"\n📁 Output directory: {OUTPUT_DIR}")

# Verify data files
print("\n📂 Checking required files:")
print(f"   Papers dataset: {PAPERS_PATH}")
if PAPERS_PATH.exists():
    print(f"      ✅ Found")
else:
    print(f"      ❌ Not found - required for processing")
    raise FileNotFoundError(f"Papers dataset not found: {PAPERS_PATH}")

print(f"   Metadata: {METADATA_PATH}")
if METADATA_PATH.exists():
    print(f"      ✅ Found")
else:
    print(f"      ❌ Not found - required for processing")
    raise FileNotFoundError(f"Metadata not found: {METADATA_PATH}")

print(f"   Checkpoint: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    print(f"      ✅ Found")
else:
    print(f"      ❌ Not found - required for inference")
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

print("\n✅ Drive setup complete")
print("="*80)

## Cell 3: Install Dependencies

In [ ]:
import subprocess

print("="*80)
print("INSTALLING DEPENDENCIES")
print("="*80)

# Install required packages
# Pin transformers to avoid PyTorch compatibility issues
packages = [
    'transformers==4.35.2',
    'datasets',
    'scikit-learn',
    'tqdm'
]

print("Installing packages (this may take 1-2 minutes)...\n")

subprocess.run(
    ['pip', 'install', '-q'] + packages,
    check=False  # Don't fail on warnings
)

print("✅ All packages installed successfully")

# Verify imports
print("\nVerifying imports...")
import transformers
import pandas as pd
import numpy as np
from tqdm import tqdm

print(f"   transformers: {transformers.__version__}")
print(f"   pandas: {pd.__version__}")
print(f"   numpy: {np.__version__}")

print("\n✅ Import verification complete")
print("="*80)

## Cell 4: Configuration and Traceability Setup

In [ ]:
import json

print("="*80)
print("CONFIGURATION & TRACEABILITY")
print("="*80)

# GPU-adaptive batch size configuration
if device.type == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        BATCH_SIZE = 128  # A100: 4x larger batches (40GB memory)
        print(f"   🚀 A100 detected - using optimized batch size: {BATCH_SIZE}")
    else:
        BATCH_SIZE = 32   # T4/other GPUs
        print(f"   💡 Using standard batch size: {BATCH_SIZE}")
else:
    BATCH_SIZE = 8  # CPU fallback

# Mixed precision configuration (A100 optimization)
USE_AMP = False
AMP_DTYPE = torch.float32

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        USE_AMP = True
        AMP_DTYPE = torch.bfloat16  # A100 has excellent bfloat16 support
        print(f"   🚀 A100 detected - enabling mixed precision (bfloat16)")
    else:
        print(f"   💡 Mixed precision disabled (not A100)")
else:
    print(f"   ℹ️  Mixed precision not available (CPU mode)")

MAX_LENGTH_CLASSIFICATION = 256  # Shorter for classification
MAX_LENGTH_NER = 512  # Longer for NER

print(f"\n⚙️  Processing Configuration:")
print(f"   Device: {device}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Mixed precision: {'✅ Enabled (bfloat16)' if USE_AMP else '❌ Disabled'}")
print(f"   Classification max length: {MAX_LENGTH_CLASSIFICATION}")
print(f"   NER max length: {MAX_LENGTH_NER}")

# Model configuration
MODEL_CONFIG = {
    'model_name_or_path': 'allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500',
    'n_metadata_features': 28,  # CRITICAL: Must match features_engineered.csv
    'num_classes': 2,  # bio-resource, not-bio-resource
    'num_ner_labels': 3,  # O, B-COM, I-COM
    'n_boolean_features': 10,
    'n_numerical_features': 4,  # NOTE: Phase 4 checkpoint uses 2, not 4. This is the target for future retraining.
    'classification_dropout': 0.3,
    'ner_dropout': 0.1
}

print(f"\n🤖 Model Configuration:")
print(f"   Base model: {MODEL_CONFIG['model_name_or_path'].split('/')[-1]}")
print(f"   Metadata features: {MODEL_CONFIG['n_metadata_features']}")
print(f"   Classification classes: {MODEL_CONFIG['num_classes']}")
print(f"   NER labels: {MODEL_CONFIG['num_ner_labels']} (O, B-COM, I-COM)")

# Verify checkpoint compatibility
print(f"\n⚠️  Model Configuration Note:")
print(f"   MODEL_CONFIG specifies n_numerical_features={MODEL_CONFIG['n_numerical_features']}")
print(f"   However, the Phase 4 checkpoint was trained with n_numerical_features=2")
print(f"   The notebook will automatically use the checkpoint's configuration")
print(f"   to ensure successful model loading.")
print(f"\n   To use all 4 numerical features, the model must be retrained.")

# Traceability configuration
TRACEABILITY_CONFIG = {
    'inference_session_id': SESSION_ID,
    'training_session_id': TRAINING_SESSION_ID,
    'checkpoint_path': str(CHECKPOINT_PATH),
    'inference_started': TIMESTAMP,
    'device': str(device),
    'batch_size': BATCH_SIZE,
    'mixed_precision': USE_AMP,
    'amp_dtype': str(AMP_DTYPE) if USE_AMP else 'float32',
    'model_config': MODEL_CONFIG,
    'dataset': {
        'papers': str(PAPERS_PATH),
        'metadata': str(METADATA_PATH)
    },
    'output_directory': str(OUTPUT_DIR)
}

print(f"\n📋 Traceability:")
print(f"   Inference Session: {SESSION_ID}")
print(f"   Training Session: {TRAINING_SESSION_ID}")
print(f"   Model Checkpoint: experiment_archives/{TRAINING_SESSION_ID}/...")

print("\n✅ Configuration complete")
print("="*80)

## Cell 5: Load Dataset Papers

In [ ]:
import pandas as pd

print("="*80)
print("LOADING DATASET")
print("="*80)

print(f"\n📥 Loading papers from: {PAPERS_PATH}")

# Load papers
papers_df = pd.read_csv(PAPERS_PATH)
print(f"✅ Loaded {len(papers_df):,} total papers")

# Check required columns
required_cols = ['id', 'title', 'abstract']
missing_cols = [col for col in required_cols if col not in papers_df.columns]
if missing_cols:
    print(f"\n❌ ERROR: Missing required columns: {missing_cols}")
    print(f"   Available columns: {list(papers_df.columns)}")
    raise ValueError(f"Missing required columns: {missing_cols}")

print(f"\n📊 Dataset Statistics:")
print(f"   Total papers: {len(papers_df):,}")
print(f"   Columns: {len(papers_df.columns)}")
print(f"   Required columns: {', '.join(required_cols)} ✅")

# Check for missing values
missing_titles = papers_df['title'].isna().sum()
missing_abstracts = papers_df['abstract'].isna().sum()
if missing_titles > 0:
    print(f"   ⚠️  Warning: {missing_titles} papers with missing titles")
if missing_abstracts > 0:
    print(f"   ⚠️  Warning: {missing_abstracts} papers with missing abstracts")

# Display sample
print(f"\n📄 Sample paper (ID: {papers_df.iloc[0]['id']}):")
print(f"   Title: {papers_df.iloc[0]['title'][:80]}...")
abstract_text = str(papers_df.iloc[0]['abstract'])
print(f"   Abstract length: {len(abstract_text)} chars")

# Update traceability
TRACEABILITY_CONFIG['dataset']['total_papers'] = len(papers_df)
TRACEABILITY_CONFIG['dataset']['papers_loaded'] = datetime.now().isoformat()

print("\n✅ Papers loaded successfully")
print("="*80)

## Cell 6: Load Metadata Features

In [ ]:
print("="*80)
print("LOADING METADATA")
print("="*80)

print(f"\n📥 Loading metadata from: {METADATA_PATH}")

# Load metadata
metadata_df = pd.read_csv(METADATA_PATH)
print(f"✅ Loaded metadata for {len(metadata_df):,} papers")

# Expected metadata features (28 total)
EXPECTED_FEATURES = [
    # Boolean indicators (10 features)
    'hasDbCrossReferences', 'hasData', 'hasSuppl', 'isOpenAccess',
    'inPMC', 'inEPMC', 'hasPDF', 'hasBook',
    'is_research_article', 'is_review_article',
    
    # Numerical features (4 features)  # FIXED: Was "2 features", should be 4
    'log_citations', 'years_since_pub', 'citedByCount', 'pubYear',
    
    # Categorical missing indicators (2 features)  # FIXED: More accurate description
    'meshTerms_missing', 'keywords_missing',
    
    # MeSH TF-IDF (7 features)
    'mesh_tfidf_0', 'mesh_tfidf_1', 'mesh_tfidf_2', 'mesh_tfidf_3',
    'mesh_tfidf_4', 'mesh_tfidf_5', 'mesh_tfidf_6',
    
    # Keyword TF-IDF (5 features)
    'keyword_tfidf_0', 'keyword_tfidf_1', 'keyword_tfidf_2',
    'keyword_tfidf_3', 'keyword_tfidf_4'
]

print(f"\n📊 Metadata Statistics:")
print(f"   Total papers with metadata: {len(metadata_df):,}")
print(f"   Expected features: {len(EXPECTED_FEATURES)}")

# Check for expected features
missing_features = [f for f in EXPECTED_FEATURES if f not in metadata_df.columns]
if missing_features:
    print(f"\n⚠️  Warning: Missing expected features: {missing_features}")
    print(f"   Model expects exactly {MODEL_CONFIG['n_metadata_features']} features")
else:
    print(f"   ✅ All {len(EXPECTED_FEATURES)} expected features present")

# Filter metadata to match papers
metadata_subset = metadata_df[metadata_df['id'].isin(papers_df['id'])].copy()
print(f"\n📊 Coverage:")
print(f"   Papers with metadata: {len(metadata_subset):,} / {len(papers_df):,}")
print(f"   Coverage: {len(metadata_subset) / len(papers_df) * 100:.1f}%")

# Check for missing metadata
missing_metadata = len(papers_df) - len(metadata_subset)
if missing_metadata > 0:
    print(f"\n⚠️  Warning: {missing_metadata} papers missing metadata")
    print("   These will use default imputation values (zeros)")

# Display feature sample
if len(metadata_subset) > 0:
    sample_id = metadata_subset.iloc[0]['id']
    print(f"\n📊 Sample metadata (ID: {sample_id}):")
    print(f"   isOpenAccess: {metadata_subset.iloc[0].get('isOpenAccess', 'N/A')}")
    print(f"   citedByCount: {metadata_subset.iloc[0].get('citedByCount', 'N/A')}")
    print(f"   pubYear: {metadata_subset.iloc[0].get('pubYear', 'N/A')}")
    print(f"   is_research_article: {metadata_subset.iloc[0].get('is_research_article', 'N/A')}")

# Update traceability
TRACEABILITY_CONFIG['dataset']['metadata_coverage'] = len(metadata_subset)
TRACEABILITY_CONFIG['dataset']['metadata_loaded'] = datetime.now().isoformat()
TRACEABILITY_CONFIG['dataset']['n_features'] = len(EXPECTED_FEATURES)

print("\n✅ Metadata loaded successfully")
print("="*80)

## Cell 7: Load Phase 4 Model

In [ ]:
import torch
from src.models.multitask_model import BiomedicalMultiTaskModel

print("="*80)
print("LOADING PHASE 4 MODEL")
print("="*80)

print(f"\n📥 Loading checkpoint: {CHECKPOINT_PATH}")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

# Extract or use default configuration
if 'config' in checkpoint:
    config = checkpoint['config']
    print("✅ Found configuration in checkpoint")
else:
    print("⚠️  No config in checkpoint, using defaults")
    config = MODEL_CONFIG.copy()

# CRITICAL: Model architecture must EXACTLY match checkpoint
# Use checkpoint config for structural parameters, MODEL_CONFIG only as fallback
print("\n🏗️  Creating model architecture...")

# For structural parameters, ALWAYS prefer checkpoint config over MODEL_CONFIG
# to ensure architecture matches saved weights
model = BiomedicalMultiTaskModel(
    model_name_or_path=config.get('model_name_or_path', MODEL_CONFIG['model_name_or_path']),
    n_metadata_features=config.get('n_metadata_features', MODEL_CONFIG['n_metadata_features']),
    num_classes=config.get('num_classes', MODEL_CONFIG['num_classes']),
    num_ner_labels=config.get('num_ner_labels', MODEL_CONFIG['num_ner_labels']),
    n_boolean_features=config.get('n_boolean_features', MODEL_CONFIG['n_boolean_features']),
    n_numerical_features=config.get('n_numerical_features', 2),  # ✅ FIXED: Use checkpoint value or fallback to 2
    classification_dropout=config.get('classification_dropout', MODEL_CONFIG['classification_dropout']),
    ner_dropout=config.get('ner_dropout', MODEL_CONFIG['ner_dropout'])
)

# Verify the architecture matches what we loaded
print(f"\n📊 Model Architecture:")
print(f"   n_metadata_features: {config.get('n_metadata_features', MODEL_CONFIG['n_metadata_features'])}")
print(f"   n_numerical_features: {config.get('n_numerical_features', 2)}")
print(f"   n_boolean_features: {config.get('n_boolean_features', MODEL_CONFIG['n_boolean_features'])}")
print(f"   num_ner_labels: {config.get('num_ner_labels', MODEL_CONFIG['num_ner_labels'])}")

# Warn if checkpoint config differs from MODEL_CONFIG
checkpoint_n_num = config.get('n_numerical_features', 2)
if checkpoint_n_num != MODEL_CONFIG['n_numerical_features']:
    print(f"\n⚠️  WARNING: Checkpoint has {checkpoint_n_num} numerical features, "
          f"but MODEL_CONFIG specifies {MODEL_CONFIG['n_numerical_features']}")
    print(f"   Using checkpoint value ({checkpoint_n_num}) to ensure weights load correctly")

# Load weights
print("\n📦 Loading model weights...")
state_dict = checkpoint.get('model_state_dict', checkpoint)

try:
    model.load_state_dict(state_dict, strict=True)
    print("✅ Loaded weights (strict mode)")
except Exception as e:
    print(f"⚠️  Strict loading failed: {e}")
    print("   Trying non-strict mode...")
    model.load_state_dict(state_dict, strict=False)
    print("✅ Loaded weights (non-strict mode)")

# Move to device and set eval mode
model.to(device)
model.eval()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📈 Model Statistics:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Model size: ~{total_params * 4 / 1024**2:.1f} MB")

# Display checkpoint metrics if available
if 'metrics' in checkpoint:
    print(f"\n🎯 Checkpoint Metrics:")
    for key, value in checkpoint['metrics'].items():
        if isinstance(value, float):
            print(f"   {key}: {value:.4f}")
        else:
            print(f"   {key}: {value}")

if 'epoch' in checkpoint:
    print(f"\n   Trained for {checkpoint['epoch']} epochs")

# Update traceability
TRACEABILITY_CONFIG['model'] = {
    'total_parameters': total_params,
    'trainable_parameters': trainable_params,
    'checkpoint_metrics': checkpoint.get('metrics', {}),
    'epoch': checkpoint.get('epoch', 'unknown'),
    'loaded_at': datetime.now().isoformat(),
    'n_numerical_features_used': checkpoint_n_num
}

print(f"\n✅ Model loaded successfully on {device}")
print("="*80)

## Cell 8: Run Classification Inference

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast
from transformers import AutoTokenizer
from tqdm import tqdm
import time

print("="*80)
print("CLASSIFICATION INFERENCE")
print("="*80)

# Load tokenizer
print("\n📚 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    'allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500'
)
print("✅ Tokenizer loaded")

# Import dataset class
from src.multitask_predict import InferenceDataset, collate_fn

# Create dataset
print("\n🔄 Creating dataset...")
classif_dataset = InferenceDataset(
    papers_df=papers_df,
    metadata_df=metadata_subset,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH_CLASSIFICATION,
    n_expected_features=MODEL_CONFIG['n_metadata_features']
)
print(f"✅ Dataset created with {len(classif_dataset)} samples")

# Note: FutureWarning from multitask_predict.py can be ignored
# This will be fixed in a future update to the inference script

# Optimized DataLoader configuration
num_workers = 2 if (torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(0)) else 0

classif_loader = DataLoader(
    classif_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=num_workers,
    pin_memory=True if device.type == 'cuda' else False,
    prefetch_factor=2 if num_workers > 0 else None,
    persistent_workers=True if num_workers > 0 else False
)
print(f"✅ DataLoader created (batch_size={BATCH_SIZE}, num_workers={num_workers})")

# Label mapping
ID2LABEL = {
    0: 'not-bio-resource',
    1: 'bio-resource'
}

# Run inference
print(f"\n🚀 Running classification inference on {len(papers_df):,} papers...")
print("   📝 Memory optimization: Storing only ID + predictions (no text duplication)")
results = []

start_time = time.time()
with torch.no_grad():
    for batch in tqdm(classif_loader, desc="Classifying"):
        # Move to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        metadata = batch['metadata'].to(device)
        
        # Forward pass with optional mixed precision
        with autocast(dtype=AMP_DTYPE, enabled=USE_AMP):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                metadata=metadata,
                task='classification',
                return_auxiliary=False
            )
        
        # Get predictions (move back to float32 for processing)
        logits = outputs['logits'].float()
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(logits, dim=-1)
        
        # Collect results - MEMORY OPTIMIZED: Store only ID and predictions
        # Text data will be merged from papers_df in Cell 10
        for i in range(len(batch['id'])):
            results.append({
                'ID': str(batch['id'][i]),
                'predicted_label': ID2LABEL[preds[i].item()],
                'probability': probs[i][1].item()  # Probability of bio-resource
            })

inference_time = time.time() - start_time

# Create results DataFrame
classification_results = pd.DataFrame(results)

# Save results
classif_output = OUTPUT_DIR / 'classification_results.csv'
classification_results.to_csv(classif_output, index=False)

print(f"\n✅ Classification complete!")
print(f"   Runtime: {inference_time:.2f}s ({int(inference_time // 60)}m {int(inference_time % 60)}s)")
print(f"   Speed: {len(classification_results) / inference_time:.1f} papers/sec")
print(f"   Papers processed: {len(classification_results):,}")

# Statistics
positive_count = (classification_results['predicted_label'] == 'bio-resource').sum()
print(f"\n📊 Classification Results:")
print(f"   Bio-resources: {positive_count:,} ({positive_count/len(classification_results)*100:.1f}%)")
print(f"   Non-resources: {len(classification_results) - positive_count:,} ({(len(classification_results)-positive_count)/len(classification_results)*100:.1f}%)")
print(f"   Mean probability: {classification_results['probability'].mean():.3f}")
print(f"   Median probability: {classification_results['probability'].median():.3f}")

print(f"\n💾 Results saved to: {classif_output}")
print(f"   Columns: {list(classification_results.columns)}")
print(f"   Memory optimized: Text data NOT stored (will merge from papers_df)")

# Update traceability
TRACEABILITY_CONFIG['classification'] = {
    'total_papers': len(classification_results),
    'bio_resources': int(positive_count),
    'runtime_seconds': inference_time,
    'papers_per_second': len(classification_results) / inference_time,
    'completed_at': datetime.now().isoformat(),
    'memory_optimized': True,
    'columns_stored': list(classification_results.columns)
}

# Store num_workers for traceability
TRACEABILITY_CONFIG['num_workers'] = num_workers

print("="*80)

## Cell 9: Run NER Inference

In [ ]:
print("="*80)
print("NER INFERENCE")
print("="*80)

# ID2TAG mapping for BIO tagging
ID2TAG = {
    0: 'O',
    1: 'B-COM',
    2: 'I-COM'
}

# Import NER utilities
from src.multitask_predict import tokens_to_words

def extract_entities_from_bio_tags(tokens, bio_tags, probabilities):
    """
    Extract entities from BIO-tagged tokens with RoBERTa special token filtering.
    
    Args:
        tokens: List of token strings
        bio_tags: List of tag IDs (0=O, 1=B-COM, 2=I-COM)
        probabilities: List of confidence scores
    
    Returns:
        List of (entity_text, entity_type, confidence) tuples
    """
    if not (len(tokens) == len(bio_tags) == len(probabilities)):
        return []
    
    entities = []
    current_entity = None
    
    for token, tag_id, prob in zip(tokens, bio_tags, probabilities):
        # Skip RoBERTa special tokens
        if token in ['<s>', '</s>', '<pad>', '<unk>']:
            continue
        
        tag = ID2TAG[tag_id]
        
        if tag.startswith('B-'):  # Beginning of entity
            if current_entity:
                entities.append(current_entity)
            
            entity_type = tag[2:]
            current_entity = {
                'tokens': [token],
                'type': entity_type,
                'probs': [prob]
            }
            
        elif tag.startswith('I-'):  # Inside entity
            if current_entity:
                entity_type = tag[2:]
                if current_entity['type'] == entity_type:
                    current_entity['tokens'].append(token)
                    current_entity['probs'].append(prob)
                else:
                    entities.append(current_entity)
                    current_entity = {
                        'tokens': [token],
                        'type': entity_type,
                        'probs': [prob]
                    }
        
        else:  # 'O' tag
            if current_entity:
                entities.append(current_entity)
                current_entity = None
    
    if current_entity:
        entities.append(current_entity)
    
    # Format entities
    formatted_entities = []
    for entity in entities:
        text = ' '.join(entity['tokens']).replace(' ##', '')
        avg_prob = sum(entity['probs']) / len(entity['probs'])
        formatted_entities.append((text, entity['type'], avg_prob))
    
    return formatted_entities

# Create NER dataset
print("\n🔄 Creating NER dataset...")
ner_dataset = InferenceDataset(
    papers_df=papers_df,
    metadata_df=metadata_subset,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH_NER,
    n_expected_features=MODEL_CONFIG['n_metadata_features']
)
print(f"✅ Dataset created with {len(ner_dataset)} samples")

# Optimized DataLoader configuration
num_workers = 2 if (torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(0)) else 0

ner_loader = DataLoader(
    ner_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=num_workers,
    pin_memory=True if device.type == 'cuda' else False,
    prefetch_factor=2 if num_workers > 0 else None,
    persistent_workers=True if num_workers > 0 else False
)
print(f"✅ DataLoader created (batch_size={BATCH_SIZE}, num_workers={num_workers})")

# Run NER inference
print(f"\n🚀 Running NER inference on {len(papers_df):,} papers...")
print("   📝 Memory optimization: Storing only ID + entities (no text duplication)")
ner_results = []
total_entities = 0

start_time = time.time()
with torch.no_grad():
    for batch in tqdm(ner_loader, desc="Extracting entities"):
        # Move to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        metadata = batch['metadata'].to(device)
        
        # Forward pass with optional mixed precision
        with autocast(dtype=AMP_DTYPE, enabled=USE_AMP):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                metadata=metadata,
                task='ner',
                return_auxiliary=False
            )
        
        # Get predictions (move back to float32 for processing)
        logits = outputs['logits'].float()
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(logits, dim=-1)
        
        # Extract entities for each sequence
        for i in range(len(batch['id'])):
            # Get tokens
            tokens = tokens_to_words(tokenizer, input_ids[i])
            
            # Get predictions and probabilities
            seq_preds = preds[i].cpu().numpy()
            seq_probs = probs[i].cpu().numpy()
            
            # Get probability of predicted tag for each token
            token_probs = [seq_probs[j, seq_preds[j]] for j in range(len(seq_preds))]
            
            # Extract entities
            entities = extract_entities_from_bio_tags(
                tokens=tokens,
                bio_tags=seq_preds.tolist(),
                probabilities=token_probs
            )
            
            total_entities += len(entities)
            
            # Separate by entity type
            com_entities = [(text, prob) for text, etype, prob in entities if etype == 'COM']
            ful_entities = [(text, prob) for text, etype, prob in entities if etype == 'FUL']
            
            # Format as comma-separated strings - MEMORY OPTIMIZED: Store only ID and entities
            # Text data will be merged from papers_df in Cell 10
            ner_results.append({
                'ID': str(batch['id'][i]),
                'common_name': ', '.join([text for text, _ in com_entities]),
                'common_prob': ', '.join([f"{prob:.3f}" for _, prob in com_entities]),
                'full_name': ', '.join([text for text, _ in ful_entities]),
                'full_prob': ', '.join([f"{prob:.3f}" for _, prob in ful_entities])
            })

ner_time = time.time() - start_time

# Create results DataFrame
ner_results_df = pd.DataFrame(ner_results)

# Save results
ner_output = OUTPUT_DIR / 'ner_results.csv'
ner_results_df.to_csv(ner_output, index=False)

print(f"\n✅ NER complete!")
print(f"   Runtime: {ner_time:.2f}s ({int(ner_time // 60)}m {int(ner_time % 60)}s)")
print(f"   Speed: {len(ner_results_df) / ner_time:.1f} papers/sec")
print(f"   Papers processed: {len(ner_results_df):,}")

# Statistics
papers_with_entities = (
    (ner_results_df['common_name'].str.strip() != '') |
    (ner_results_df['full_name'].str.strip() != '')
).sum()
papers_with_com = (ner_results_df['common_name'].str.strip() != '').sum()
papers_with_ful = (ner_results_df['full_name'].str.strip() != '').sum()

print(f"\n📊 NER Results:")
print(f"   Total entities extracted: {total_entities:,}")
print(f"   Papers with entities: {papers_with_entities:,} ({papers_with_entities/len(ner_results_df)*100:.1f}%)")
print(f"   Papers with COM entities: {papers_with_com:,}")
print(f"   Papers with FUL entities: {papers_with_ful:,}")
print(f"   Avg entities per paper: {total_entities / len(ner_results_df):.2f}")

print(f"\n💾 Results saved to: {ner_output}")
print(f"   Columns: {list(ner_results_df.columns)}")
print(f"   Memory optimized: Text data NOT stored (will merge from papers_df)")

# Update traceability
TRACEABILITY_CONFIG['ner'] = {
    'total_papers': len(ner_results_df),
    'papers_with_entities': int(papers_with_entities),
    'total_entities': int(total_entities),
    'runtime_seconds': ner_time,
    'papers_per_second': len(ner_results_df) / ner_time,
    'completed_at': datetime.now().isoformat(),
    'memory_optimized': True,
    'columns_stored': list(ner_results_df.columns)
}

print("="*80)

## Cell 10: Create Final Inventory and Merge Results

In [ ]:
import gc
import psutil

def get_memory_usage():
    """Get current memory usage in GB"""
    process = psutil.Process()
    return process.memory_info().rss / 1024**3

print("="*80)
print("MERGING RESULTS & CREATING FINAL INVENTORY (MEMORY OPTIMIZED)")
print("="*80)

initial_memory = get_memory_usage()
print(f"\n📊 Initial memory usage: {initial_memory:.2f} GB")

# Ensure classification_results has 'ID'
if 'ID' not in classification_results.columns:
    classification_results.rename(columns={'id': 'ID'}, inplace=True)

# Ensure ner_results_df has 'ID'
if 'ID' not in ner_results_df.columns:
    ner_results_df.rename(columns={'id': 'ID'}, inplace=True)

# Prepare metadata
if len(metadata_subset) > 0:
    metadata_cols = ['id', 'isOpenAccess', 'citedByCount', 'pubYear',
                     'is_research_article', 'is_review_article']
    available_cols = [col for col in metadata_cols if col in metadata_subset.columns]

    if len(available_cols) > 1:
        metadata_for_merge = metadata_subset[available_cols].copy()
        
        # CRITICAL FIX #1: Filter NaN BEFORE converting to string
        metadata_before = len(metadata_for_merge)
        metadata_for_merge = metadata_for_merge[metadata_for_merge['id'].notna()].copy()
        metadata_filtered = metadata_before - len(metadata_for_merge)
        
        if metadata_filtered > 0:
            print(f"⚠️  Filtered {metadata_filtered} metadata rows with NaN IDs")
        
        metadata_for_merge.rename(columns={'id': 'ID'}, inplace=True)
        # NOW convert to string (after filtering NaN)
        metadata_for_merge['ID'] = metadata_for_merge['ID'].astype(str)
        print("✅ Prepared metadata for merge")
    else:
        metadata_for_merge = None
        print("ℹ️  No additional metadata to merge")
else:
    metadata_for_merge = None
    print("ℹ️  No metadata found")

# CRITICAL FIX #2: Filter NaN from papers_df BEFORE converting to string
papers_before = len(papers_df)
if 'id' in papers_df.columns:
    papers_df = papers_df[papers_df['id'].notna()].copy()
    papers_filtered = papers_before - len(papers_df)
    if papers_filtered > 0:
        print(f"⚠️  Filtered {papers_filtered} papers with NaN IDs")
    papers_df.rename(columns={'id': 'ID'}, inplace=True)
elif 'ID' in papers_df.columns:
    papers_df = papers_df[papers_df['ID'].notna()].copy()
    papers_filtered = papers_before - len(papers_df)
    if papers_filtered > 0:
        print(f"⚠️  Filtered {papers_filtered} papers with NaN IDs")
else:
    raise ValueError("Base papers_df missing 'id' or 'ID' column")

# NOW convert ID to string (after filtering NaN)
papers_df['ID'] = papers_df['ID'].astype(str)

print(f"✅ Prepared all secondary DataFrames for merging.")
print(f"   Papers (after NaN filter): {len(papers_df):,} rows")
print(f"   Classification results: {len(classification_results):,} rows")
print(f"   NER results: {len(ner_results_df):,} rows")
if metadata_for_merge is not None:
    print(f"   Metadata (after NaN filter): {len(metadata_for_merge):,} rows")

# Process in chunks
CHUNK_SIZE = 5000
total_rows = len(papers_df)
num_chunks = (total_rows + CHUNK_SIZE - 1) // CHUNK_SIZE
final_output = OUTPUT_DIR / 'final_inventory.csv'

print(f"\n🔄 Processing {total_rows:,} papers in {num_chunks} chunks (size={CHUNK_SIZE:,})...")
print(f"   Using papers_df as base (contains text data)")
print(f"   Merging slim classification and NER results onto each chunk")

# For statistics
total_bio_resources = 0
total_papers_with_entities = 0

# Determine column order
base_cols = list(papers_df.columns)
classif_cols = [c for c in classification_results.columns if c != 'ID']
ner_cols = [c for c in ner_results_df.columns if c != 'ID']
meta_cols = [c for c in metadata_for_merge.columns if c != 'ID'] if metadata_for_merge is not None else []
final_columns = base_cols + classif_cols + ner_cols + meta_cols

first_chunk = True

for i in range(0, total_rows, CHUNK_SIZE):
    chunk_end = min(i + CHUNK_SIZE, total_rows)
    chunk_num = i // CHUNK_SIZE + 1
    print(f"   Chunk {chunk_num}/{num_chunks}: rows {i:,}-{chunk_end:,}")

    base_chunk = papers_df.iloc[i:chunk_end].copy()
    merged_chunk = pd.merge(base_chunk, classification_results, on='ID', how='left')
    merged_chunk = pd.merge(merged_chunk, ner_results_df, on='ID', how='left')
    
    if metadata_for_merge is not None:
        merged_chunk = pd.merge(merged_chunk, metadata_for_merge, on='ID', how='left')

    total_bio_resources += (merged_chunk['predicted_label'] == 'bio-resource').sum()
    papers_with_entities_chunk = (
        (merged_chunk['common_name'].fillna('').str.strip() != '') |
        (merged_chunk['full_name'].fillna('').str.strip() != '')
    ).sum()
    total_papers_with_entities += papers_with_entities_chunk

    merged_chunk.to_csv(final_output, mode='a' if not first_chunk else 'w',
                       header=first_chunk, index=False, columns=final_columns)
    first_chunk = False

    del base_chunk, merged_chunk
    gc.collect()

    current_memory = get_memory_usage()
    print(f"      Memory: {current_memory:.2f} GB (+{current_memory - initial_memory:.2f} GB)")

print(f"\n✅ Final inventory saved: {final_output}")

print(f"\n📊 Final Statistics:")
print(f"   Total rows: {total_rows:,}")
print(f"   Bio-resources: {total_bio_resources:,} ({total_bio_resources/total_rows*100:.1f}%)")
print(f"   Papers with entities: {total_papers_with_entities:,} ({total_papers_with_entities/total_rows*100:.1f}%)")

# Store stats for Cell 11
positive_count = total_bio_resources
papers_with_entities = total_papers_with_entities
if 'ner' in TRACEABILITY_CONFIG and 'total_entities' in TRACEABILITY_CONFIG['ner']:
    total_entities = TRACEABILITY_CONFIG['ner']['total_entities']
    print(f"   Total entities extracted: {total_entities:,}")

# Update traceability
TRACEABILITY_CONFIG['final_inventory'] = {
    'total_papers': int(total_rows),
    'total_columns': len(final_columns),
    'created_at': datetime.now().isoformat(),
    'chunked_processing': True,
    'chunk_size': CHUNK_SIZE,
    'memory_optimized': True,
    'merge_strategy': 'papers_df_base_with_slim_results',
    'nan_filtering': 'applied_before_merge'
}

# Clean up
del papers_df, classification_results, ner_results_df
if metadata_for_merge is not None:
    del metadata_for_merge
gc.collect()

final_memory = get_memory_usage()
print(f"\n✅ Memory cleaned up. Current usage: {final_memory:.2f} GB")
print(f"   Peak memory increase: +{final_memory - initial_memory:.2f} GB")
print("✅ Final inventory created successfully")
print("="*80)

## Cell 11: Create Documentation and Archive

In [ ]:
import json

print("="*80)
print("CREATING DOCUMENTATION & ARCHIVE")
print("="*80)

# Calculate total runtime
total_runtime = inference_time + ner_time
completion_time = datetime.now()
TRACEABILITY_CONFIG['inference_completed'] = completion_time.isoformat()
TRACEABILITY_CONFIG['total_runtime_seconds'] = total_runtime

# Save traceability configuration
print("\n💾 Saving configuration with traceability...")
config_output = OUTPUT_DIR / 'config_with_traceability.json'
with open(config_output, 'w') as f:
    json.dump(TRACEABILITY_CONFIG, f, indent=2)
print(f"   ✅ Saved: {config_output}")

# Create README using stored statistics
print("\n📝 Creating README...")

# Get counts from stored variables (set at end of Cell 10)
total_papers_count = TRACEABILITY_CONFIG['final_inventory']['total_papers']
bio_resources_count = positive_count
papers_with_entities_count = papers_with_entities
total_entities_count = total_entities if 'total_entities' in locals() else TRACEABILITY_CONFIG['ner']['total_entities']

# Get classification and NER counts from traceability
classif_papers = TRACEABILITY_CONFIG['classification']['total_papers']
ner_papers = TRACEABILITY_CONFIG['ner']['total_papers']

readme_content = f"""# Phase 4 Multi-Task Inference Results

**Session ID**: {SESSION_ID}
**Training Session**: {TRAINING_SESSION_ID}
**Created**: {TIMESTAMP}
**Completed**: {completion_time.strftime('%Y-%m-%d %H:%M:%S')}
**Total Runtime**: {int(total_runtime // 60)}m {int(total_runtime % 60)}s

---

## Overview

Full 2022 dataset inference using Phase 4 unified multi-task model.

## Dataset

- **Source**: {PAPERS_PATH}
- **Papers**: {total_papers_count:,}
- **Metadata**: {METADATA_PATH}
- **Features**: {MODEL_CONFIG['n_metadata_features']}

## Model

- **Checkpoint**: {CHECKPOINT_PATH}
- **Architecture**: Phase 4 BiomedicalMultiTaskModel
- **Base Model**: {MODEL_CONFIG['model_name_or_path'].split('/')[-1]}
- **Parameters**: {total_params:,}
- **Device**: {device}

## Results

### Classification

- **Total Papers**: {classif_papers:,}
- **Bio-resources**: {bio_resources_count:,} ({bio_resources_count/classif_papers*100:.1f}%)
- **Runtime**: {int(inference_time // 60)}m {int(inference_time % 60)}s
- **Speed**: {classif_papers / inference_time:.1f} papers/sec

### NER

- **Total Papers**: {ner_papers:,}
- **Papers with Entities**: {papers_with_entities_count:,} ({papers_with_entities_count/ner_papers*100:.1f}%)
- **Total Entities**: {total_entities_count:,}
- **Avg per Paper**: {total_entities_count / ner_papers:.2f}
- **Runtime**: {int(ner_time // 60)}m {int(ner_time % 60)}s
- **Speed**: {ner_papers / ner_time:.1f} papers/sec

## Output Files

1. **classification_results.csv**: Classification predictions with probabilities
2. **ner_results.csv**: Named entity extraction results
3. **final_inventory.csv**: Merged results with metadata
4. **config_with_traceability.json**: Complete configuration and traceability
5. **README.md**: This file

## Traceability

Full traceability chain from training to inference:

- **Training Session**: {TRAINING_SESSION_ID}
- **Inference Session**: {SESSION_ID}
- **Checkpoint**: {CHECKPOINT_PATH}
- **Configuration**: config_with_traceability.json

## Performance vs V2 Baseline

Phase 4 achieves:
- **NER F1**: 0.9274 (+23.82% vs V2 baseline 0.749)
- **Classification F1**: 0.8586 (-4.38% vs V2 baseline 0.898)
- **Combined F1**: 0.8917 (+8.28% vs V2 baseline 0.8235)
- **Runtime**: ~{int(total_runtime // 60)} minutes (vs ~45 minutes for V2)

## Next Steps

1. Download results from Google Drive
2. Compare with V2 baseline results
3. Proceed with downstream URL extraction and name processing
4. Update inventory database

---

**Generated by**: phase4_full_inference_2022.ipynb
**Documentation**: docs/multi_task_model/PHASE4_IMPLEMENTATION_SUMMARY.md
"""

readme_output = OUTPUT_DIR / 'README.md'
with open(readme_output, 'w') as f:
    f.write(readme_content)
print(f"   ✅ Saved: {readme_output}")

# Output already in experiment_archives
print("\n📦 Results Location:")
print(f"   Primary output: {OUTPUT_DIR}")
print(f"   All results saved to: experiment_archives/{SESSION_ID}_phase4_2022_rerun/")

# Create a symbolic reference in collab_results for backward compatibility
collab_results_dir = Path('collab_results')
collab_results_dir.mkdir(exist_ok=True)

# Create a clearly-named pointer file
reference_file = collab_results_dir / f'MOVED_TO_EXPERIMENT_ARCHIVES_{SESSION_ID}.txt'
with open(reference_file, 'w') as f:
    f.write(f"="*80 + "\n")
    f.write(f"RESULTS LOCATION FOR SESSION {SESSION_ID}\n")
    f.write(f"="*80 + "\n\n")
    f.write(f"Results for this session have been saved to:\n")
    f.write(f"  experiment_archives/{SESSION_ID}_phase4_2022_rerun/\n\n")
    f.write(f"All Phase 4 inference runs now save directly to experiment_archives/\n")
    f.write(f"for better organization and archival.\n\n")
    f.write(f"Files in this directory:\n")
    f.write(f"  - classification_results.csv\n")
    f.write(f"  - ner_results.csv\n")
    f.write(f"  - final_inventory.csv\n")
    f.write(f"  - config_with_traceability.json\n")
    f.write(f"  - README.md\n")

print(f"   ✅ Reference pointer created: {reference_file.name}")

print("\n✅ Documentation and archive created successfully")
print("="*80)

## Cell 12: Summary and Next Steps

In [ ]:
print("="*80)
print("🎉 PHASE 4 INFERENCE COMPLETE!")
print("="*80)

print(f"\n🆔 Session Information:")
print(f"   Inference Session ID: {SESSION_ID}")
print(f"   Training Session ID: {TRAINING_SESSION_ID}")
print(f"   Started: {TIMESTAMP}")
print(f"   Completed: {completion_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Total Runtime: {int(total_runtime // 60)}m {int(total_runtime % 60)}s")

# Get counts from stored variables and traceability
total_papers_count = TRACEABILITY_CONFIG['final_inventory']['total_papers']
classif_papers = TRACEABILITY_CONFIG['classification']['total_papers']
ner_papers = TRACEABILITY_CONFIG['ner']['total_papers']
bio_resources_count = positive_count
papers_with_entities_count = papers_with_entities
total_entities_count = total_entities if 'total_entities' in locals() else TRACEABILITY_CONFIG['ner']['total_entities']

print(f"\n📊 Processing Summary:")
print(f"   Papers Processed: {total_papers_count:,}")
print(f"   Device: {device}")
print(f"   Batch Size: {BATCH_SIZE}")

print(f"\n🎯 Results Summary:")
print(f"   Classification:")
print(f"      Bio-resources: {bio_resources_count:,} ({bio_resources_count/classif_papers*100:.1f}%)")
print(f"      Runtime: {int(inference_time // 60)}m {int(inference_time % 60)}s")
print(f"      Speed: {classif_papers / inference_time:.1f} papers/sec")

print(f"\n   NER:")
print(f"      Papers with entities: {papers_with_entities_count:,} ({papers_with_entities_count/ner_papers*100:.1f}%)")
print(f"      Total entities: {total_entities_count:,}")
print(f"      Runtime: {int(ner_time // 60)}m {int(ner_time % 60)}s")
print(f"      Speed: {ner_papers / ner_time:.1f} papers/sec")

print(f"\n💾 Output Files:")
print(f"   📁 {OUTPUT_DIR}")
print(f"      ├── classification_results.csv ({classif_papers:,} rows)")
print(f"      ├── ner_results.csv ({ner_papers:,} rows)")
print(f"      ├── final_inventory.csv ({total_papers_count:,} rows)")
print(f"      ├── config_with_traceability.json")
print(f"      └── README.md")

print(f"\n📦 Archive Location:")
print(f"   experiment_archives/{SESSION_ID}_phase4_2022_rerun/")

print(f"\n🔗 Traceability Chain:")
print(f"   Training → {TRAINING_SESSION_ID}")
print(f"   Checkpoint → {CHECKPOINT_PATH}")
print(f"   Inference → {SESSION_ID}")
print(f"   Archive → experiment_archives/{SESSION_ID}_phase4_2022_rerun/")

print(f"\n📈 Performance vs V2 Baseline:")
print(f"   NER F1: 0.9274 (+23.82% improvement)")
print(f"   Classification F1: 0.8586 (-4.38% trade-off)")
print(f"   Combined F1: 0.8917 (+8.28% overall improvement)")
print(f"   Runtime: ~{int(total_runtime // 60)}min (vs ~45min for V2)")

print(f"\n📝 Next Steps:")
print(f"   1. ✅ Download results from Google Drive")
print(f"   2. ⏳ Compare with V2 baseline using comparison scripts")
print(f"   3. ⏳ Run downstream URL extraction (src/url_extractor.py)")
print(f"   4. ⏳ Run name processing (src/process_names.py)")
print(f"   5. ⏳ Update final biodata inventory")

print(f"\n🎊 Success! Phase 4 inference completed in {int(total_runtime // 60)}m {int(total_runtime % 60)}s")
print(f"🆔 Session ID: {SESSION_ID}")
print("="*80)